# A small OTLiNGAM tutorial

This notebook generates a linear non-Gaussian system, learns a causal order, and compares the true and estimated weighted adjacency matrices. OTLiNGAM uses one-dimensional Wasserstein distance to measure how far standardized residuals are from a standard Gaussian.

The method is described in [Contrast-Free ICA and Causal Inference via Wasserstein Distances to the Gaussian](https://arxiv.org/abs/2607.12832).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from otlingam import GreedyOTLiNGAM

## Generate a non-Gaussian SEM

Rows are children and columns are parents. The data follow `X = (I - B)^{-T} e` in row-observation form.

In [ ]:
rng = np.random.default_rng(42)
n_samples = 5_000
adjacency = np.array(
    [
        [0.0, 0.0, 0.0, 0.0, 0.0],
        [0.8, 0.0, 0.0, 0.0, 0.0],
        [0.0, -0.7, 0.0, 0.0, 0.0],
        [0.5, 0.0, 0.9, 0.0, 0.0],
        [0.0, -0.6, 0.0, 0.7, 0.0],
    ]
)
noise = rng.uniform(-1.0, 1.0, size=(n_samples, adjacency.shape[0]))
X = noise @ np.linalg.inv(np.eye(adjacency.shape[0]) - adjacency).T
X.shape

## Learn the order

The greedy estimator is a good starting point when the number of variables is not tiny. It repeatedly chooses the variable whose standardized residual is most non-Gaussian.

In [ ]:
model = GreedyOTLiNGAM().fit(X)
print("Estimated order:", model.causal_order_)
print("Score:", model.score_)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout="constrained")
for ax, matrix, title in zip(
    axes,
    (adjacency, model.adjacency_matrix_),
    ("True adjacency", "Estimated adjacency"),
    strict=True,
): 
    image = ax.imshow(matrix, cmap="RdBu_r", vmin=-1.0, vmax=1.0)
    ax.set_title(title)
    ax.set_xlabel("Parent")
    ax.set_ylabel("Child")
fig.colorbar(image, ax=axes, label="Edge weight")
plt.show()

## Choosing another estimator

`ExhaustiveOTLiNGAM` searches all subset states and is useful for small systems. Its exponential cost makes `GreedyOTLiNGAM` the practical default as the dimension grows. `OTICALiNGAM` provides an ICA-LiNGAM route with OTICA source estimation.